# RewardBench2 Multi-Probe Debiasing

Evaluates a reward model on RewardBench2 while simultaneously applying all available bias probes from `artifacts/probes/`. The workflow is:

 1. load and dimension-check every `.pt` probe;
 2. average probes within each bias category (length, position, sycophancy, uncertainty) and orthonormalise them with Gram–Schmidt;
 3. in a single forward pass compute baseline rewards, per-probe-nullified rewards, and a combined-nullified score;
 4. report per-subset accuracy with binomial 95 % CIs and save a bar-chart summary.

 The probe must beat **all** rejected responses to count as correct.

 **REMARKS:**

 - This experiment can be quite slow.
 - Ideally, probes and reward model should share base architecture and weights.


In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import sys, yaml, os

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset


In [ ]:
# Clone ideally the `multigpu` branch
!git clone -b feature/multigpu https://github.com/nondatur/OneBiasAfterAnotherFork.git

In [ ]:
# We specify these two global variables:
HF_HOME   = "/content/huggingface"
REPO_HOME = "/content/OneBiasAfterAnotherFork"

In [ ]:
# Set up HF home (for Colab only)
os.environ['HF_HOME'] = HF_HOME
if HF_HOME not in sys.path:
    sys.path.append(HF_HOME)

# Ensure the repository root is in the system path
repo_path = os.path.abspath(REPO_HOME)
if repo_path not in sys.path:
     sys.path.append(repo_path)

# Set up the project root
PROJECT_ROOT = Path(REPO_HOME).resolve()
if str(PROJECT_ROOT) not in sys.path:
     sys.path.insert(0, str(PROJECT_ROOT))

# Check system path(s)
for path in sys.path:
    print(f"Current sys.path(s): {path}")

In [ ]:
from src.nb.nullbias.probe import ( # type: ignore
    get_base_model,
    get_score_head,
    tokenize_inputs,
    gram_schmidt,
    project_to_null_space,
)

In [ ]:
def load_probes(probes_dir: Path) -> Dict[str, torch.Tensor]:
    """Load all probes from the probes directory.

    Returns:
        Dictionary mapping probe name to probe tensor
        Probes can be 1D vectors [hidden_dim] or 2D matrices [n_basis, hidden_dim]
    """
    probes = {}

    for bias_type_dir in probes_dir.iterdir():
        if not bias_type_dir.is_dir():
            continue

        for probe_dir in bias_type_dir.iterdir():
            probe_path = probe_dir / "probe.pt"
            if probe_path.exists():
                probe = torch.load(probe_path, map_location="cpu")
                name = f"{bias_type_dir.name}/{probe_dir.name}"
                probes[name] = probe
                if probe.dim() == 1:
                    print(f"Loaded probe: (shape={list(probe.shape)}, dim={probe.shape[0]})",
                          name)
                elif probe.dim() == 2:
                    print(f"Loaded probe: (shape={list(probe.shape)}, n_basis={probe.shape[0]}, hidden_dim={probe.shape[1]})",
                          name)
                else:
                    print(f"Loaded probe: (unexpected shape={list(probe.shape)})",
                          name)

    return probes


In [ ]:
def get_rewards_multi(
    model: AutoModelForSequenceClassification,
    tokenizer: AutoTokenizer,
    texts: List[str],
    probes: Dict[str, torch.Tensor],
    combined_basis: Optional[torch.Tensor] = None,
    null_alpha: float = 1.0,
    batch_size: int = 8,
    device: str = "cuda",
    max_length: int = 2048,
    show_progress: bool = True,
) -> Dict[str, torch.Tensor]:
    """Compute baseline, individual probe, and combined rewards in a single forward pass.

    Args:
        model: Reward model
        tokenizer: Tokenizer
        texts: List of formatted texts
        probes: Dict mapping probe name to probe tensor
        combined_basis: [n_probes, hidden_dim] orthonormal basis for all probes combined
        null_alpha: Nullification strength
        batch_size: Batch size
        device: Device for inputs
        max_length: Max sequence length
        show_progress: Show progress bar

    Returns:
        Dict mapping condition name to rewards tensor:
        - "baseline": no nulling
        - "all_probes": all probes combined
        - "{probe_name}": each individual probe
    """
    model.eval()
    base_model = get_base_model(model)
    score_head = get_score_head(model)

    # Prepare probes - handle both 1D vectors and 2D matrices
    probe_vecs = {}
    for name, probe in probes.items():
        probe = probe.to(device).float()
        if probe.dim() == 1:
            # Single vector probe
            probe_vecs[name] = probe / (probe.norm() + 1e-8)
        elif probe.dim() == 2:
            # Matrix probe (e.g., position probe with multiple basis vectors)
            # Use the first basis vector as the probe direction for individual evaluation
            v = probe[0]
            probe_vecs[name] = v / (v.norm() + 1e-8)
        else:
            raise ValueError(f"Unexpected probe shape for {name}: {probe.shape}")

    if combined_basis is not None and combined_basis.shape[0] > 0:
        combined_basis = combined_basis.to(device).float()

    # Pre-tokenize all texts (handles both strings and pairs)
    print("Tokenizing %d texts..." % (len(texts)))
    all_encodings = tokenize_inputs(
        tokenizer,
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    # Initialize score collectors
    all_scores = {name: [] for name in ["baseline", "all_probes"] + list(probes.keys())}

    n_batches = (len(texts) + batch_size - 1) // batch_size
    iterator = range(n_batches)
    if show_progress:
        iterator = tqdm(iterator, desc="Computing rewards", total=n_batches)

    with torch.no_grad():
        for batch_idx in iterator:
            start = batch_idx * batch_size
            end = min(start + batch_size, len(texts))

            inputs = {k: v[start:end].to(device) for k, v in all_encodings.items()}

            # Get hidden states once
            outputs = base_model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]

            attention_mask = inputs["attention_mask"]
            last_token_indices = attention_mask.sum(dim=1) - 1
            last_hidden = hidden_states[
                torch.arange(hidden_states.size(0), device=device),
                last_token_indices,
            ].float()

            # Baseline score (no nulling)
            baseline_scores = score_head(last_hidden.to(hidden_states.dtype)).squeeze(-1)
            all_scores["baseline"].append(baseline_scores.cpu())

            # All probes combined
            if combined_basis is not None and combined_basis.shape[0] > 0:
                nulled_hidden = project_to_null_space(last_hidden, combined_basis)
                combined_scores = score_head(nulled_hidden.to(hidden_states.dtype)).squeeze(-1)
            else:
                combined_scores = baseline_scores
            all_scores["all_probes"].append(combined_scores.cpu())

            # Each probe individually
            for name, probe_vec in probe_vecs.items():
                proj = (last_hidden @ probe_vec).unsqueeze(-1)
                nulled = last_hidden - null_alpha * proj * probe_vec.unsqueeze(0)
                scores = score_head(nulled.to(hidden_states.dtype)).squeeze(-1)
                all_scores[name].append(scores.cpu())

    return {name: torch.cat(scores_list, dim=0) for name, scores_list in all_scores.items()}


In [ ]:
def load_rewardbench(split: str = "test") -> List[Dict]:
    """Load RewardBench 2 dataset.

    RewardBench 2 has multiple rejected responses per example.
    The model must score chosen higher than ALL rejected to be correct.

    The prompt field is a list of messages (conversation history).
    """
    print(f"Loading RewardBench 2 (split={split})")
    dataset = load_dataset("allenai/reward-bench-2", split=split)

    examples = []
    for idx, row in enumerate(dataset):
        prompt = row.get("prompt", [])
        chosen_raw = row.get("chosen", "")
        rejected_list = row.get("rejected", [])

        # prompt is a list of messages in RewardBench 2
        # Convert to string if it's a list
        if isinstance(prompt, list):
            # It's a conversation - extract the text content
            if prompt and isinstance(prompt[0], dict):
                # List of message dicts
                prompt_text = "\n\n".join(
                    m.get("content", "") for m in prompt if m.get("content")
                )
            else:
                # List of strings
                prompt_text = "\n\n".join(str(p) for p in prompt)
        else:
            prompt_text = str(prompt)

        # chosen is a list in RewardBench 2, extract first element
        if isinstance(chosen_raw, list):
            chosen = chosen_raw[0] if chosen_raw else ""
        else:
            chosen = chosen_raw

        # Ensure rejected is a list
        if isinstance(rejected_list, str):
            rejected_list = [rejected_list]

        if prompt_text and chosen and rejected_list:
            examples.append({
                "idx": idx,
                "prompt": prompt_text,
                "prompt_raw": prompt,  # Keep original for proper formatting
                "chosen": chosen,
                "rejected": rejected_list,  # List of rejected responses
                "subset": row.get("subset", "unknown"),
            })

    print(f"Loaded {len(examples)} RewardBench 2 examples")
    return examples


In [ ]:
def format_rb2_conversation(tokenizer: Any, prompt_raw: Any, response: str):
    """Format RewardBench 2 conversation with response.

    Handles the case where prompt is a list of messages.

    Args:
        tokenizer: HuggingFace tokenizer
        prompt_raw: Raw prompt (list of messages or string)
        response: Assistant response to append

    Returns:
        Formatted conversation string OR tuple (prompt, response) for pair-format models
    """
    # Extract prompt text first
    if isinstance(prompt_raw, list):
        if prompt_raw and isinstance(prompt_raw[0], dict):
            prompt_text = "\n\n".join(m.get("content", "") for m in prompt_raw)
        else:
            prompt_text = "\n\n".join(str(p) for p in prompt_raw)
    else:
        prompt_text = str(prompt_raw)

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        # Build conversation from prompt messages
        if isinstance(prompt_raw, list) and prompt_raw and isinstance(prompt_raw[0], dict):
            # Already in message format
            conv = list(prompt_raw)
        elif isinstance(prompt_raw, list):
            # List of strings - treat as user messages
            conv = [{"role": "user", "content": prompt_text}]
        else:
            conv = [{"role": "user", "content": prompt_text}]

        # Add assistant response
        conv.append({"role": "assistant", "content": response})

        formatted = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        # Remove BOS token - it will be added back during tokenization
        if tokenizer.bos_token is not None and formatted.startswith(tokenizer.bos_token):
            formatted = formatted[len(tokenizer.bos_token):]
        return formatted
    else:
        # Pair format for models without chat template (e.g., DeBERTa)
        # Return tuple for tokenizer(prompt, response) pair encoding
        return (prompt_text, response)


Evaluation of results:

In [ ]:
def evaluate_rewardbench_multi(
    model: AutoModelForSequenceClassification,
    tokenizer: AutoTokenizer,
    examples: List[Dict],
    probes: Dict[str, torch.Tensor],
    probe_basis: Optional[torch.Tensor] = None,
    null_alpha: float = 1.0,
    batch_size: int = 8,
    device: str = "cuda",
    max_length: int = 2048,
) -> Dict[str, Dict[str, float]]:
    """Evaluate on RewardBench 2 with baseline, individual probes, and all combined.

    All computed in a single forward pass for efficiency.

    For each example, chosen must score higher than ALL rejected responses.

    Returns:
        Dict mapping condition name to results dict:
        - "baseline": no nulling
        - "all_probes": all probes combined
        - "{probe_name}": each individual probe
    """
    # Prepare all texts - flatten rejected lists for batch scoring
    all_chosen = []
    all_rejected = []  # Flat list of all rejected texts
    rejected_counts = []  # Number of rejected per example
    subsets = []

    for ex in examples:
        prompt_raw = ex.get("prompt_raw", ex["prompt"])
        all_chosen.append(format_rb2_conversation(tokenizer, prompt_raw, ex["chosen"]))
        rejected_list = ex["rejected"]
        rejected_counts.append(len(rejected_list))
        for rej in rejected_list:
            all_rejected.append(format_rb2_conversation(tokenizer, prompt_raw, rej))
        subsets.append(ex["subset"])

    # Get rewards for chosen (all conditions)
    print("Computing chosen rewards (%d texts)..." % (len(all_chosen)))
    chosen_rewards = get_rewards_multi(
        model, tokenizer, all_chosen,
        probes=probes,
        combined_basis=probe_basis,
        null_alpha=null_alpha,
        batch_size=batch_size,
        device=device,
        max_length=max_length,
    )

    # Get rewards for all rejected (all conditions)
    print("Computing rejected rewards (%d texts)..." % (len(all_rejected)))
    rejected_rewards = get_rewards_multi(
        model, tokenizer, all_rejected,
        probes=probes,
        combined_basis=probe_basis,
        null_alpha=null_alpha,
        batch_size=batch_size,
        device=device,
        max_length=max_length,
    )

    def compute_accuracy(chosen_r, rejected_r):
        """Compute accuracy from reward tensors."""
        correct = []
        rej_idx = 0
        for i, count in enumerate(rejected_counts):
            ch = chosen_r[i]
            rej = rejected_r[rej_idx : rej_idx + count]
            rej_idx += count
            is_correct = (ch > rej).all().item()
            correct.append(float(is_correct))
        return torch.tensor(correct)

    def build_results(correct):
        """Build results dict from correctness tensor."""
        results = {
            "accuracy": correct.mean().item(),
            "n_examples": len(examples),
            "total_rejected": len(all_rejected),
            "avg_rejected_per_example": len(all_rejected) / len(examples),
        }

        subset_correct = {}
        subset_total = {}
        for i, subset in enumerate(subsets):
            if subset not in subset_correct:
                subset_correct[subset] = 0
                subset_total[subset] = 0
            subset_correct[subset] += correct[i].item()
            subset_total[subset] += 1

        for subset in sorted(subset_correct.keys()):
            results[f"accuracy_{subset}"] = subset_correct[subset] / subset_total[subset]
            results[f"n_{subset}"] = subset_total[subset]

        return results

    # Compute results for all conditions
    all_results = {}
    for condition in chosen_rewards.keys():
        correct = compute_accuracy(chosen_rewards[condition], rejected_rewards[condition])
        all_results[condition] = build_results(correct)

    return all_results

In [ ]:
# Multiplier for 95% confidence interval
CI_95_MULTIPLIER = 1.96

def binomial_ci95(p: float, n: int) -> float:
    """Compute 95% confidence interval half-width for a proportion."""
    if n <= 0:
        return 0.0
    se = np.sqrt(p * (1 - p) / n)
    return CI_95_MULTIPLIER * se


In [ ]:
def create_rewardbench_multiprobe_plot(
    baseline: Dict[str, float],
    combined: Dict[str, float],
    individual: Dict[str, Dict[str, float]],
    output_path: Path,
    title: str = "RewardBench 2 Results",
) -> None:
    """Create bar plot showing baseline, individual probes, and combined accuracy.

    Args:
        baseline: Baseline results dict
        combined: Combined (all probes) results dict
        individual: Dict mapping probe name to results dict
        output_path: Path to save the plot
        title: Plot title
    """
    plt.style.use('seaborn-v0_8-whitegrid')

    # Prepare data: baseline, individual probes, combined
    conditions = ["Baseline"]
    accuracies = [baseline["accuracy"] * 100]
    n = baseline["n_examples"]

    # Add individual probes
    for name in sorted(individual.keys()):
        # Shorten name
        short = name.split("/")[-1] if "/" in name else name
        # Further shorten if needed
        if len(short) > 20:
            short = short[:17] + "..."
        conditions.append(short)
        accuracies.append(individual[name]["accuracy"] * 100)

    # Add combined
    conditions.append("All Combined")
    accuracies.append(combined["accuracy"] * 100)

    # Compute error bars
    errors = [binomial_ci95(acc/100, n) * 100 for acc in accuracies]

    # Compute deltas from baseline
    baseline_acc = accuracies[0]
    deltas = [acc - baseline_acc for acc in accuracies]

    # Create plot
    fig, ax = plt.subplots(figsize=(max(10, len(conditions) * 0.9), 6))

    x = np.arange(len(conditions))

    # Color scheme: baseline is blue, individual are grey, combined is green
    colors = ["#4C72B0"]  # Baseline
    colors.extend(["#888888"] * len(individual))  # Individual probes
    colors.append("#55A868")  # Combined

    bars = ax.bar(x, accuracies, yerr=errors, capsize=3,
                  color=colors, edgecolor="#333", linewidth=0.5)

    # Add value labels and delta on bars
    for i, (bar, delta) in enumerate(zip(bars, deltas)):
        height = bar.get_height()
        # Value label
        ax.annotate(f"{height:.1f}%",
                   xy=(bar.get_x() + bar.get_width()/2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha="center", va="bottom", fontsize=8)
        # Delta label (skip baseline)
        if i > 0:
            delta_color = "green" if delta >= 0 else "red"
            ax.annotate(f"({delta:+.1f})",
                       xy=(bar.get_x() + bar.get_width()/2, height + errors[i] + 3),
                       xytext=(0, 3), textcoords="offset points",
                       ha="center", va="bottom", fontsize=7, color=delta_color)

    ax.set_ylabel("Accuracy (%)", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(conditions, rotation=45, ha="right", fontsize=9)
    ax.set_ylim(0, max(accuracies) + 15)

    # Title
    full_title = f"{title}\n(n={n}, {len(individual)} probes)"
    ax.set_title(full_title, fontsize=13, fontweight="bold")

    # Add reference line at baseline
    ax.axhline(y=baseline_acc, color="#4C72B0", linestyle="--", alpha=0.5, linewidth=1)

    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()


In [ ]:
def _load_defaults(section: str) -> dict:
    """Return default config values for *section* from configs/default_values.yaml."""
    with open(PROJECT_ROOT / "configs" / "default_values.yaml") as f:
        import yaml as _yaml
        return _yaml.safe_load(f).get(section, {})

In [ ]:
def run(config_path: "str | Path | None" = None, **overrides):
    """Evaluate a reward model on RewardBench 2 with multi-probe debiasing.

    Args:
        config_path: Path to a YAML config.  If relative, resolved from the
                     project-root ``configs/`` directory.
        **overrides: Keyword overrides, e.g. ``device="cpu"``, ``null_alpha=0.5``.
    """
    cfg = _load_defaults("rewardbench_multiprobe")

    if config_path is not None:
        p = Path(config_path)
        if not p.is_absolute():
            p = PROJECT_ROOT / "configs" / p
        with open(p) as f:
            cfg.update(yaml.safe_load(f))

    cfg.update(overrides)

    cfg.update(overrides)

    # Load probes
    all_probes = load_probes(Path(cfg["probes_dir"]))

    if not all_probes:
        print(f"No probes found in {Path(cfg['probes_dir'])}")
        print("Run bias experiments first to generate probes!")
        sys.exit(1)

    # Filter probes by model name if specified
    if cfg.get("probe_model"):
        filtered = {k: v for k, v in all_probes.items() if cfg.get("probe_model").lower() in k.lower()}
        if not filtered:
            print(f"No probes found for model '{cfg.get('probe_model')}'. Available probes:")
            for name in all_probes:
                print(f"  - {name}")
            sys.exit(1)
        all_probes = filtered
        print(f"Filtered to probes for model: {cfg.get('probe_model')}")

    # Filter probes by specific names if specified
    if cfg.get("probes"):
        filtered = {}
        for name in cfg.get("probes"):
            matches = [k for k in all_probes if name in k]
            for m in matches:
                filtered[m] = all_probes[m]
        all_probes = filtered

    print(cfg)

    # Load model first to get hidden dimension
    print(f"Loading model: {cfg['model']}")
    tokenizer = AutoTokenizer.from_pretrained(cfg["model"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model"],
        trust_remote_code=True,
        dtype=torch.bfloat16,
    )
    model = model.to(cfg["device"])

    # Get model's hidden dimension
    base_model = get_base_model(model)
    hidden_dim = base_model.config.hidden_size
    print(f"Model hidden dimension: {hidden_dim}")

    # Filter probes to only those compatible with model's hidden dimension
    # and expand matrices into individual vectors
    compatible_probes = {}
    for name, probe in all_probes.items():
        if probe.dim() == 1:
            # 1D vector: check if dimension matches
            if probe.shape[0] == hidden_dim:
                compatible_probes[name] = probe
            else:
                print(f"Skipping probe {name}: dimension mismatch ({probe.shape[0]} != {hidden_dim})")
        elif probe.dim() == 2:
            # 2D matrix: check if hidden_dim matches
            if probe.shape[1] == hidden_dim:
                # For matrices, we'll use all basis vectors separately
                # Add each basis vector as a separate probe
                for i in range(probe.shape[0]):
                    vec_name = f"{name}_basis{i}" if probe.shape[0] > 1 else name
                    compatible_probes[vec_name] = probe[i]
            else:
                print(f"Skipping probe {name}: hidden_dim mismatch ({probe.shape[1]} != {hidden_dim})")
        else:
            print(f"Skipping probe {name}: unexpected shape {list(probe.shape)}")

    all_probes = compatible_probes

    print(f"Using {len(all_probes)} compatible probes:")
    for name in all_probes:
        print(f"  - {name}")

    # Group probes by category and average within each category
    # Categories: sycophancy, uncertainty, position, length
    category_probes = {}  # category -> list of normalized probes
    for name, probe in all_probes.items():
        # Extract category from probe name (e.g., "sycophancy/sycophancy_gemma2_mmlu" -> "sycophancy")
        category = name.split("/")[0] if "/" in name else "other"
        probe = probe.float()
        probe = probe / (probe.norm() + 1e-8)  # Normalize
        if category not in category_probes:
            category_probes[category] = []
        category_probes[category].append(probe)

    # Average probes within each category, then normalize
    averaged_probes = {}
    for category, probes in category_probes.items():
        if len(probes) == 1:
            averaged = probes[0]
        else:
            # Stack and average
            stacked = torch.stack(probes, dim=0)  # [n, hidden_dim]
            averaged = stacked.mean(dim=0)  # [hidden_dim]
        # Re-normalize the averaged probe
        averaged = averaged / (averaged.norm() + 1e-8)
        averaged_probes[category] = averaged
        print(f"Category '{category}': averaged {len(probes)} probes into 1 direction")

    # Stack averaged category probes for combined nulling
    probe_list = list(averaged_probes.values())
    probe_basis = torch.stack(probe_list, dim=0)  # [n_categories, hidden_dim]
    print(f"Combined basis: {probe_basis.shape[0]} category directions (averaged)")

    # Load RewardBench 2
    examples = load_rewardbench(cfg["split"])

    # Evaluate all conditions in single pass
    print("\n" + "=" * 60)
    print(f"EVALUATING (baseline + {len(all_probes)} probes individually + all combined)")
    print("=" * 60)
    all_results = evaluate_rewardbench_multi(
        model, tokenizer, examples,
        probes=all_probes,
        probe_basis=probe_basis,
        null_alpha=cfg["null_alpha"],
        batch_size=cfg["batch_size"],
        device=cfg["device"],
        max_length=cfg["max_length"],
    )

    baseline_results = all_results["baseline"]
    combined_results = all_results["all_probes"]

    # Print summary table
    print("\n" + "=" * 80)
    print("OVERALL ACCURACY BY CONDITION")
    print("=" * 80)
    print("%-50s %10s %10s" % ("Condition", "Accuracy", "Delta"))
    print("-" * 80)

    baseline_acc = baseline_results["accuracy"]
    print("%-50s %9.2f%% %10s" % ("Baseline (no nulling)", 100 * baseline_acc, "-"))

    # Individual probes
    individual_results = {}
    for name in sorted(all_probes.keys()):
        if name in all_results:
            acc = all_results[name]["accuracy"]
            delta = acc - baseline_acc
            individual_results[name] = all_results[name]
            # Shorten probe name for display
            short_name = name.split("/")[-1] if "/" in name else name
            print("%-50s %9.2f%% %+9.2f%%" % (f"  {short_name}", 100 * acc, 100 * delta))

    # Combined
    combined_acc = combined_results["accuracy"]
    delta_combined = combined_acc - baseline_acc
    print("-" * 80)
    print("%-50s %9.2f%% %+9.2f%%" % ("All probes combined", 100 * combined_acc, 100 * delta_combined))

    # Print per-subset breakdown for baseline vs combined
    print("\n" + "=" * 80)
    print("SUBSET BREAKDOWN: Baseline vs All Probes Combined")
    print("=" * 80)
    print("%-30s %10s %10s %10s" % ("Subset", "Baseline", "Combined", "Delta"))
    print("-" * 80)

    print("%-30s %9.2f%% %9.2f%% %+9.2f%%" % (
                "OVERALL", 100 * baseline_acc, 100 * combined_acc, 100 * delta_combined))

    for key in sorted(baseline_results.keys()):
        if key.startswith("accuracy_") and not key.startswith("accuracy_n"):
            subset = key[9:]
            b_acc = baseline_results[key]
            c_acc = combined_results.get(key, 0)
            delta = c_acc - b_acc
            print("%-30s %9.2f%% %9.2f%% %+9.2f%%" % (
                        subset, 100 * b_acc, 100 * c_acc, 100 * delta))

    # Save results
    Path(cfg["output"]).parent.mkdir(parents=True, exist_ok=True)
    results = {
        "probes_used": list(all_probes.keys()),
        "n_probes": len(all_probes),
        "null_alpha": cfg["null_alpha"],
        "baseline": baseline_results,
        "all_probes": combined_results,
        "individual_probes": individual_results,
    }
    with open(Path(cfg["output"]), "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {cfg['output']}")

    # Generate plot with individual probes
    plot_path = Path(cfg["plots_dir"]) / f"rewardbench2_{cfg.get('probe_model') or 'all'}_plot.png"
    create_rewardbench_multiprobe_plot(
        baseline_results,
        combined_results,
        individual_results,
        output_path=plot_path,
        title=f"RewardBench 2: {cfg.get('probe_model') or 'All'} Probes",
    )
    print(f"Plot saved to {plot_path}")

In [ ]:
run("rewardbench_deberta.yaml")
